<a href="https://colab.research.google.com/github/siddharthsinh-dev/iu-bsc-thesis-llm-comparison/blob/main/notebooks/exp1_d_gemma3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Thesis**: A Comparative Study of Large Language Models for Financial Sentiment Analysis and Their Predictive Potential for Short-Term Stock Price Movement

**Component: Experiment 1** - Model D : Google DeepMind Gemma 3 (4B Instruct)

**Description:** This notebook evaluates Gemma 3 4B on the Financial PhraseBank dataset under zero-shot conditions. It generates sentiment predictions and computes accuracy, precision, recall, F1-score (macro-averaged), and confusion matrix.

Select T4 GPU as runtime.

In [ ]:
# Step 1 — Install bitsandbytes (restart required after this)

!pip install -q -U bitsandbytes accelerate

Go to runtime -> restart this session again -> then run cell 1 and run cell 2

In [ ]:
# Import required libraries — Experiment 1d: Gemma 3

import pandas as pd
import numpy as np
import torch
import warnings
import gc

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected    : {torch.cuda.get_device_name(0)}")

Add the API Key from Hugging Face using "Add New Secret" in Google Colab

In [ ]:
# Connect to Hugging Face and load Financial PhraseBank

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

print("Hugging Face login successful.")

# Load Financial PhraseBank — full dataset
file_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    filename="sentences_50agree/train-00000-of-00001.parquet",
    repo_type="dataset",
    revision="0dd3028d70cbd18ded8887e65e83343b03a50482",
    token=hf_token
)

df_fpb = pd.read_parquet(file_path)

label_map = {0: "negative", 1: "neutral", 2: "positive"}
df_fpb["sentiment"] = df_fpb["label"].map(label_map)

true_labels = df_fpb["sentiment"].tolist()
texts = df_fpb["sentence"].tolist()

print(f"\nFinancial PhraseBank loaded.")
print(f"Total sentences : {len(df_fpb)}")
print(f"Label distribution:")
print(df_fpb["sentiment"].value_counts())

In [ ]:
# Load Gemma 3 4B

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Gemma 3 4B...")

gemma_tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-4b-it",
    token=hf_token
)

gemma_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-4b-it",
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Gemma 3 4B loaded successfully.")

In [ ]:
# Define Gemma 3 classifier and run on full dataset

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

def clean_label(label):
    label = label.strip().lower()
    if "positive" in label:
        return "positive"
    elif "negative" in label:
        return "negative"
    elif "neutral" in label:
        return "neutral"
    else:
        return "neutral"

def classify_gemma(text):
    messages = [
        {"role": "user", "content": f"You are a financial sentiment classifier. Classify the sentiment of this financial text into exactly one word: positive, negative, or neutral.\n\nText: {text}\n\nSentiment:"}
    ]

    inputs = gemma_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(gemma_model.device)

    with torch.no_grad():
        outputs = gemma_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]
    generated = gemma_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )
    return clean_label(generated)

# Run on full dataset
gemma_preds = []
total = len(texts)

print(f"Running Gemma 3 on {total} sentences...")

for i, text in enumerate(texts):
    label = classify_gemma(text)
    gemma_preds.append(label)

    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{total}")

print(f"\nDone. Total predictions: {len(gemma_preds)}")
print(f"\nPrediction distribution:")
print(pd.Series(gemma_preds).value_counts())

In [ ]:
# Evaluate Gemma 3 performance on full dataset

labels_order = ["positive", "negative", "neutral"]

acc  = accuracy_score(true_labels, gemma_preds)
prec = precision_score(true_labels, gemma_preds, average="macro", labels=labels_order)
rec  = recall_score(true_labels, gemma_preds, average="macro", labels=labels_order)
f1   = f1_score(true_labels, gemma_preds, average="macro", labels=labels_order)

print("=" * 45)
print("Gemma 3 4B — Experiment 1 Results")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)

print("\nDetailed Classification Report:")
print(classification_report(true_labels, gemma_preds, labels=labels_order))

In [ ]:
# Save Gemma 3 results

gemma_scores = {
    "model": "Gemma 3 4B",
    "accuracy": round(acc, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "f1_score": round(f1, 4)
}

df_fpb["gemma_pred"] = gemma_preds

print("Gemma 3 4B — Results Summary")
print(pd.DataFrame([gemma_scores]))

In [ ]:
# Save Gemma 3 predictions to Google Drive

from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/Thesis_Data", exist_ok=True)

df_fpb.to_csv("/content/drive/MyDrive/Thesis_Data/exp1_gemma_preds.csv", index=False)

print("Gemma 3 predictions saved to Google Drive.")

In [ ]:
# Confusion Matrix — Gemma 3 4B Experiment 1

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels_order = ["positive", "negative", "neutral"]

cm = confusion_matrix(true_labels, gemma_preds, labels=labels_order)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order,
            yticklabels=labels_order)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Gemma 3 4B — Confusion Matrix (Experiment 1)", fontsize=12)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/Thesis_Data/fig_cm_gemma.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Confusion matrix saved to Google Drive.")